# 🩺 MedGraphRAG — **POC** (reduced scale · real data · NVIDIA NIM · optional GPU)

A faithful, *small-scale* end-to-end run of the **complete** hybrid Graph-RAG pipeline — every
methodological contribution preserved (C1 RGD · C2 CA-RRF · C3 CARe), every stage present — sized
for **Kaggle's free tier**. Inference runs on **NVIDIA NIM** (embeddings · reranking · RAGAS judge ·
entity extraction · generation), with an optional **GPU generation backend**.

> The original full-detail notebook (`medgraphrag_kaggle.ipynb`) is untouched. This is a separate,
> POC-dedicated notebook.

---

### ⚠️ Revision — what changed after the 2026-07-13 run, and why

That run completed, but **none of C1/C2/C3 were actually exercised**. Each was inert for a
specific reason, all of them now fixed. Read this before re-running:

| Symptom in the 2026-07-13 run | Real cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'mgr.data'` | `.gitignore` had an unanchored `data/`, which also matched the `mgr/data/` **package** — it was never committed | fixed in the repo; **the notebook must no longer write shim modules.** §3 now fails loudly instead |
| CA-RRF changed top-5 on **0/16** queries | graph had **1 concept over 60 chunks** — toy UMLS is a 10-term list, so query concepts were empty and CA-RRF degenerated to plain RRF | `UMLS_SOURCE="scispacy"` is now the default, with numpy pinned **before** first import (§0) |
| reranker `HTTP 404` → token-overlap fallback | `/v1/ranking` is the *self-hosted container* path; the hosted API uses `/v1/retrieval/{model}/reranking` | fixed in `mgr/clients/nim.py` (tries both) |
| 2 of 6 arms `Failed`, no reason shown | one over-long question per arm hit `HTTP 400 Input length …`; the item scored **wrong** and only the last error survived | retrieval query capped in `mgr/generate/executor.py`; every item error kept; §14 now prints them |
| `HTTP 429` aborted Stage 5 | repo backoff used *full* jitter from a 1.0 s base, so retries landed in the same window | equal-jitter backoff from 3.0 s in `mgr/clients/openai_compat.py`. **The notebook no longer needs a retry monkey-patch** |
| CARe frontier looked perfect (quality 1.000) | `q_with`/`q_wo` were **simulated** with `rng.random()`, and gate labels were a threshold on two of the gate's own input features — circular | §18 now *measures* rerank-on vs rerank-off answer correctness |
| RAGAS faithfulness 0.15 | the answer prompt contained the retrieved context but **not the question**, and was capped at 8 tokens | §15 passes the question and uses its own token budget |
| No-RAG tied for best | all 16 items were `anatomy-*` (unshuffled head slice) answered against the first 600 chunks of *Harrison's Internal Medicine* — the corpus could not answer them | §8 stratifies questions across MMLU subjects and samples the corpus across **several** textbooks |

**Reporting note:** the 2026-07-13 headline was Hybrid-CARRF-CARe **−0.062** vs No-RAG
(95% CI [−0.188, +0.000], Cohen's d **−0.250**) — i.e. *below* No-RAG. Any writeup quoting `+0.062`
has the sign inverted.

---

### This version adds three "real-data" upgrades (all optional, all with safe fallbacks)
1. **Real benchmark + corpus** — a **MIRAGE** question subset (the MedRAG benchmark) + a real
   **MedRAG medical-textbook** corpus slice, sampled across several books. Falls back to a
   synthetic set if offline.
2. **GPU generation option** — `GEN_BACKEND="hf_local"` runs a small HuggingFace instruct model
   (default `Qwen/Qwen2.5-3B-Instruct`) on the Kaggle GPU; NIM still does embeddings/rerank/judge.
3. **Real UMLS grounding** — `UMLS_SOURCE="scispacy"` links entities to real **UMLS CUIs** via
   scispaCy's UMLS knowledge base, feeding the repo's `UMLSLinker` + coverage curve (F5). Falls back
   to a toy dictionary — but that fallback **invalidates C2 and C3**, so it now warns loudly.

### Accelerator guidance
- `GEN_BACKEND="nim"` (default): **CPU is enough** — NIM generates remotely.
- `GEN_BACKEND="hf_local"`: turn **Accelerator → GPU T4** on (a 3B model fits comfortably).

### Kaggle setup
1. **Settings → Internet → ON** (clone, NIM, MIRAGE/HF downloads).
2. **Add-ons → Secrets →** add `NIM_API_KEY` (free at **build.nvidia.com**, starts `nvapi-`).
3. GPU only if using `hf_local`.
4. **Run §0 first, on its own.** It may ask for one kernel restart — that is expected and is the
   whole reason scispaCy works this time.

## 0 · Dependencies **first** — run this cell alone, before anything else

scispaCy requires `numpy<2`; Kaggle ships numpy 2.x. Installing scispaCy *after* numpy has been
imported downgrades the package on disk while the old module object stays in memory — which is what
produced `No module named 'numpy.rec'` and `No module named 'numpy.core._methods'` last time, and
what pushed the previous run onto the toy UMLS dictionary.

So: every install happens here, **before the first `import numpy` anywhere in the notebook**. If
numpy was already loaded, this cell stops and asks for one restart (*Run → Restart & clear cell
outputs*), then you re-run from here. It is a one-time cost.

In [ ]:
# ---- §0 · installs BEFORE any numpy import -------------------------------- #
import subprocess, sys, importlib.metadata as md

WANT_SCISPACY = True          # set False to skip real UMLS (invalidates C2/C3 — see §11)
SCISPACY_MODEL_URL = ("https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/"
                      "releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz")

def sh(cmd):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, text=True)

def version(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return None

_numpy_before = sys.modules.get("numpy")
_numpy_ver_before = getattr(_numpy_before, "__version__", None)

sh(f"{sys.executable} -m pip install -q openpyxl pyyaml duckdb pyarrow matplotlib tqdm pandas "
   f"datasets huggingface_hub")

if WANT_SCISPACY:
    # Pin numpy first so pip resolves scispaCy's tree against it in one solve,
    # rather than installing scispaCy and letting it drag numpy backwards after.
    sh(f'{sys.executable} -m pip install -q "numpy<2" "scipy<1.13"')
    sh(f"{sys.executable} -m pip install -q scispacy")
    sh(f"{sys.executable} -m pip install -q {SCISPACY_MODEL_URL}")

_numpy_ver_after = version("numpy")
print(f"\nnumpy on disk: {_numpy_ver_after}   (was imported as: {_numpy_ver_before})")

if _numpy_before is not None and _numpy_ver_before != _numpy_ver_after:
    raise SystemExit(
        "\n" + "=" * 78 +
        "\nSTOP — numpy changed underneath a module that was already imported."
        f"\n  in memory: {_numpy_ver_before}    on disk: {_numpy_ver_after}"
        "\n\nRestart the kernel (Run -> Restart & clear cell outputs), then run this"
        "\ncell again. It will be a no-op the second time and the notebook continues."
        "\n" + "=" * 78
    )
print("Dependencies ready — numpy was not imported before this cell. Continue.")

## 1 · Configuration

All scale knobs, data sources, NIM settings, the generation backend, and the UMLS source live here.
Without a `NIM_API_KEY` the notebook falls back to deterministic offline stand-ins so it still runs.

In [ ]:
from dataclasses import dataclass, asdict
from typing import Tuple

@dataclass
class POCConfig:
    # ---- repository ----
    REPO_URL: str    = "https://github.com/SLIMIHamda/clinical_graphrag_study.git"
    REPO_BRANCH: str = "main"
    REPO_DIR: str    = ""
    RUN_DIR: str     = ""

    # ---- data source ----
    DATA_SOURCE: str = "real"                 # "real" (MIRAGE + MedRAG textbooks) | "synthetic"
    MIRAGE_URL: str  = "https://raw.githubusercontent.com/Teddy-XiongGZ/MIRAGE/main/benchmark.json"
    MIRAGE_DATASET: str = "mmlu"              # mmlu | medqa | medmcqa | pubmedqa | bioasq
    MIRAGE_STRATIFY: bool = True              # spread items across subjects, not a head slice.
                                              # False reproduces the 2026-07-13 run: the first 16
                                              # MMLU items are ALL `anatomy-*`.
    CORPUS_HF: str   = "MedRAG/textbooks"     # real medical corpus (StatPearls needs MedRAG's NCBI build)
    # Several books, not one. MMLU-Med spans anatomy / biochemistry / physiology /
    # genetics / clinical knowledge; answering all of it out of Harrison's Internal
    # Medicine alone is why every retrieval arm lost to No-RAG last time.
    CORPUS_BOOKS: Tuple[str, ...] = (
        "Anatomy_Gray", "Biochemistry_Lippincott", "Cell_Biology_Molecular",
        "Physiology_Levy", "Pathology_Robbins", "Pharmacology_Katzung",
        "InternalMed_Harrison", "First_Aid_Step1",
    )
    N_CORPUS_CHUNKS: int = 2400               # total across books (BM25 + dense index).
                                              # Embedded in batches of 32 and cached to
                                              # cache/corpus_emb.npy, so this is ~75 NIM
                                              # requests once, not per run.
    CORPUS_SAMPLE: str = "spread"             # "spread" = stride through each book (skips front
                                              # matter) | "head" = first-N, the old behaviour
    GRAPH_MAX_CHUNKS: int = 800               # chunks fed to the scispaCy graph build.
                                              # This is the runtime driver (~1 s/chunk, local
                                              # CPU). It also sets how much of the corpus
                                              # carries concepts: at 120/1600 the graph covered
                                              # 7% of passages, so CA-RRF's concept boost could
                                              # not reach the documents dense/BM25 returned.

    # ---- experiment ----
    BACKBONE: str  = "Llama-70B"              # NOTE: this selects the *manifest row*. Actual
                                              # generation is GEN_MODEL below — §20 records both
                                              # and warns when they disagree.
    SEEDS: Tuple[int, ...] = (42,)
    N_ITEMS: int   = 64                       # 1 item = 100/N accuracy points, so N=64 puts the
                                              # resolution at 1.6 pts. At N=16 the entire
                                              # -0.062 headline was a ONE-item difference.
                                              # ~15 NIM requests/item at 40 rpm -> ~25 min.
    CONDITIONS: Tuple[str, ...] = (
        "No-RAG", "BM25", "Dense-MedCPT", "Graph-only", "Hybrid-CARRF", "Hybrid-CARRF-CARe",
    )
    BENCHMARK: str = "MMLU-Med"               # set automatically from MIRAGE_DATASET when DATA_SOURCE="real"

    # ---- generation backend ----
    GEN_BACKEND: str = "nim"                  # "nim" | "hf_local" (GPU) | "offline"
    HF_MODEL_ID: str = "Qwen/Qwen2.5-3B-Instruct"   # medical option: "BioMistral/BioMistral-7B" (+4bit)
    HF_LOAD_4BIT: bool = False
    MAX_NEW_TOKENS: int = 8                   # enough for an MCQ letter
    RAGAS_MAX_NEW_TOKENS: int = 160           # §15 needs prose, not a letter
    TEMPERATURE: float = 0.0

    # ---- NVIDIA NIM (embeddings / rerank / judge / extraction, and nim generation) ----
    USE_NIM: bool     = True
    NIM_BASE_URL: str = "https://integrate.api.nvidia.com"   # no trailing /v1
    NIM_RPM: int      = 40
    GEN_MODEL: str    = "meta/llama-3.1-8b-instruct"
    EMB_MODEL: str    = "nvidia/nv-embed-v1"
    RERANK_MODEL: str = "nvidia/nv-rerankqa-mistral-4b-v3"
    JUDGE_MODEL: str  = "meta/llama-3.1-8b-instruct"

    # ---- UMLS grounding ----
    # "scispacy" = real UMLS CUIs. "toy" is a 10-term keyword list: against a real
    # textbook corpus it links ~0.7% of mentions and yields a 1-concept graph, which
    # silently reduces CA-RRF to plain RRF and leaves C2/C3 unmeasured. Not equivalent.
    UMLS_SOURCE: str  = "scispacy"            # "scispacy" (real UMLS) | "toy"
    SCISPACY_MODEL: str = "en_core_sci_sm"
    UMLS_MAX_CONCEPTS: int = 1500
    UMLS_COVERAGE_FLOOR: float = 0.20         # G3 go/no-go: below this the graph is too sparse
                                              # for the concept signal to do anything

    # ---- evaluation scale ----
    RAGAS_SUBSET: int = 12
    CARE_ORACLE_ITEMS: int = 64               # items labelled by measured rerank-on vs -off (§12).
                                              # Costs up to 2 generation calls + 1 rerank each.
    CTX_TOP_K: int = 3                        # passages placed in the prompt
    N_BOOT: int = 5_000
    N_PERM: int = 20_000

    # ---- control ----
    SEED: int = 0
    FORCE: bool = False

CFG = POCConfig()
print("POC configuration:")
for k, v in asdict(CFG).items():
    print(f"  {k:22s} = {v!r}")

## 2 · Environment, paths, seeds, logging & NIM key

In [ ]:
import os, sys, json, logging, random
from pathlib import Path
import numpy as np

KAGGLE = Path("/kaggle/working").is_dir()
BASE = Path("/kaggle/working") if KAGGLE else Path("./mgr_poc").resolve()
if not CFG.REPO_DIR: CFG.REPO_DIR = str(BASE / "clinical_graphrag_study")
if not CFG.RUN_DIR:  CFG.RUN_DIR  = str(BASE / "poc_runs")
RUN = Path(CFG.RUN_DIR)
DIRS = {k: RUN / k for k in ["data", "results", "figures", "artifacts", "logs", "checkpoints", "cache"]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

random.seed(CFG.SEED); np.random.seed(CFG.SEED)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(DIRS["logs"] / "poc.log")], force=True)
log = logging.getLogger("poc")

NIM_KEY = os.environ.get("NIM_API_KEY", "")
if not NIM_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        NIM_KEY = UserSecretsClient().get_secret("NIM_API_KEY")
    except Exception as e:
        log.info(f"Kaggle Secrets unavailable ({e}); relying on env var.")
HAVE_NIM = bool(NIM_KEY) and CFG.USE_NIM

try:
    import torch; HAS_GPU = torch.cuda.is_available(); GPU = torch.cuda.get_device_name(0) if HAS_GPU else "none"
except Exception:
    HAS_GPU, GPU = False, "torch-not-installed"

print(f"Kaggle env : {KAGGLE}")
print(f"GPU        : {GPU}")
print(f"NIM key    : {'present' if NIM_KEY else 'MISSING -> offline fallback'}")
print(f"Services   : {'NIM' if HAVE_NIM else 'OFFLINE'} | generation backend requested: {CFG.GEN_BACKEND}")

## 3 · Clone the repository & preflight

Dependencies are already installed (§0). This cell only clones the repo and then **verifies the
clone is complete**.

> **Do not add compatibility shims here.** The 2026-07-13 run hand-wrote `mgr/data/loader.py` and
> `mgr/data/convert_mirage.py` into the cloned tree because they were missing. That masked the real
> bug (an unanchored `data/` in `.gitignore` excluded the package from every commit) and introduced
> new ones — the shim's `load_items` returned `[]` where the real one raises `FileNotFoundError`,
> so runs came back `n_items=0` while the notebook printed *"Completed 6 runs"*. The package is
> committed now. If this preflight fails, fix the repo, don't patch around it here.

In [ ]:
import shutil, sys
from pathlib import Path

# `sh` comes from §0. Redefined here so resuming mid-notebook after the §0
# kernel restart raises nothing more confusing than it needs to.
if "sh" not in dir():
    import subprocess
    def sh(cmd):
        print("$", cmd)
        return subprocess.run(cmd, shell=True, text=True)

repo = Path(CFG.REPO_DIR)
if not (repo / "pyproject.toml").exists():
    r = sh(f"git clone --depth 1 -b {CFG.REPO_BRANCH} {CFG.REPO_URL} '{repo}'")
    if r.returncode != 0:
        cand = list(Path("/kaggle/input").glob("**/pyproject.toml")) if Path("/kaggle/input").exists() else []
        if cand: shutil.copytree(cand[0].parent, repo, dirs_exist_ok=True); print("Used dataset copy:", cand[0].parent)
        else: raise RuntimeError("Clone failed; enable Internet or attach the repo as a Dataset.")
else:
    print("Repo already present:", repo)

if CFG.GEN_BACKEND == "hf_local":
    sh(f"{sys.executable} -m pip install -q transformers accelerate")
    if CFG.HF_LOAD_4BIT: sh(f"{sys.executable} -m pip install -q bitsandbytes")

if str(repo) not in sys.path: sys.path.insert(0, str(repo))

# ---- preflight: the clone must be complete. Fail loudly, never shim. -------- #
REQUIRED = [
    "mgr/__init__.py", "mgr/data/__init__.py", "mgr/data/loader.py",
    "mgr/data/convert_mirage.py", "mgr/generate/executor.py",
    "mgr/clients/nim.py", "mgr/retrieval/ca_rrf.py", "mgr/rerank/care_gate.py",
    "manifest/manifest.py",
]
missing = [p for p in REQUIRED if not (repo / p).exists()]
if missing:
    raise RuntimeError(
        "Incomplete clone — these files are absent from the cloned tree:\n  "
        + "\n  ".join(missing)
        + "\n\nThis is a REPO bug, not a notebook bug. Most likely a .gitignore pattern is"
        "\nmatching a source directory (an unanchored `data/` matches `mgr/data/`). Check with:"
        "\n    git check-ignore -v mgr/data/loader.py"
        "\n    git archive HEAD | tar -t | grep mgr/data"
        "\nFix and push the repo. Do NOT write replacement modules in this notebook: the"
        "\nprevious run did that and the guessed `load_items` returned [] instead of raising,"
        "\nwhich made six empty runs look successful."
    )

import mgr, manifest  # noqa: F401
from mgr.data.loader import load_items          # noqa: F401
from mgr.data.convert_mirage import convert_records  # noqa: F401
import inspect
_sig = inspect.signature(load_items)
assert list(_sig.parameters)[:2] == ["benchmark", "data_root"], f"unexpected load_items signature: {_sig}"
print("Imported mgr + manifest from", repo)
print("Preflight OK — mgr.data present, load_items signature is", _sig)

## 4 · Inference backends — NIM services + generation factory

`build_services()` returns the injectable services the pipeline consumes. **Embeddings, reranker,
judge, and entity-extractor always come from NIM** (the repo's `nim_adapters`) when a key is present.
**Generation** is chosen by `GEN_BACKEND`:

- `nim` → small hosted NIM model (repo's low-volume permitted lane; the `NimClient.chat` guard that
  blocks the *bulk* sweep is respected — we use a separate rate-limited client for tiny POC volume).
- `hf_local` → a small HuggingFace instruct model on the **Kaggle GPU** (NIM still does the rest).
- `offline` → deterministic stand-in (no key / dry run).

In [ ]:
import re, hashlib
import numpy as np
from mgr.clients.openai_compat import OpenAICompatClient

_SAFE_GEN = {"temperature", "top_p", "max_tokens", "seed"}
def _toks(s): return re.findall(r"[a-z0-9]+", s.lower())

class NimGenerationClient:
    def __init__(self, base_url, api_key, model, rpm=40, temperature=0.0, max_tokens=8):
        self._c = OpenAICompatClient(base_url=base_url, api_key=api_key, rpm=rpm)
        self.model, self.temperature, self.max_tokens = model, temperature, max_tokens
    def complete_text(self, model, messages, **params):
        p = {k: v for k, v in params.items() if k in _SAFE_GEN}
        p.setdefault("temperature", self.temperature); p.setdefault("max_tokens", self.max_tokens)
        r = self._c.chat(self.model, messages, **p); u = r.get("usage", {})
        return (r["choices"][0]["message"]["content"],
                {"in": int(u.get("prompt_tokens", 0)), "out": int(u.get("completion_tokens", 0))})

class LocalHFClient:
    """Small HuggingFace instruct model on the Kaggle GPU. Implements complete_text."""
    def __init__(self, model_id, max_new_tokens=8, temperature=0.0, load_4bit=False):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch
        self.tok = AutoTokenizer.from_pretrained(model_id)
        kw = dict(torch_dtype="auto", device_map="auto" if torch.cuda.is_available() else None)
        if load_4bit:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
        self.max_new_tokens, self.temperature = max_new_tokens, temperature
    def complete_text(self, model, messages, **params):
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = self.tok(prompt, return_tensors="pt").to(self.model.device)
        out = self.model.generate(**enc, max_new_tokens=self.max_new_tokens,
              do_sample=self.temperature > 0, temperature=max(self.temperature, 1e-5),
              pad_token_id=self.tok.eos_token_id)
        gen = out[0][enc["input_ids"].shape[1]:]
        return self.tok.decode(gen, skip_special_tokens=True), {"in": int(enc["input_ids"].shape[1]), "out": int(gen.shape[0])}

# ---- offline deterministic stand-ins (same interfaces) ----
class OfflineGen:
    def complete_text(self, model, messages, **params):
        u = messages[-1]["content"]; ti = max(1, len(u)//4); ctx = ""
        if "Context:" in u: ctx = u.split("Context:", 1)[1].split("Question:", 1)[0]
        opts = dict(re.findall(r"^([A-D])\.\s*(.+)$", u, flags=re.M))
        if opts:
            ls = list(opts.keys())
            if ctx.strip():
                cs = set(_toks(ctx)); ov = lambda x: len(set(_toks(x)) & cs)
                b = max(ls, key=lambda L: (ov(opts[L]), L))
                if ov(opts[b]) > 0: return b, {"in": ti, "out": 1}
            h = int(hashlib.sha256(u.encode()).hexdigest(), 16); return ls[h % len(ls)], {"in": ti, "out": 1}
        if "yes, no, or maybe" in u: return "maybe", {"in": ti, "out": 1}
        if "yes or no" in u: return "yes", {"in": ti, "out": 1}
        return "unknown", {"in": ti, "out": 1}
class OfflineEmbedder:
    dim = 64
    def __call__(self, texts):
        out = np.zeros((len(texts), self.dim))
        for i, t in enumerate(texts):
            for w in _toks(t): out[i, int(hashlib.md5(w.encode()).hexdigest(), 16) % self.dim] += 1.0
        return out
class OfflineJudge:
    def decompose(self, text): return [s.strip() for s in re.split(r"[.\n]", text) if s.strip()]
    def entails(self, h, p): return len(set(_toks(h)) & set(_toks(p))) >= 1
    def relevant(self, q, p): return len(set(_toks(q)) & set(_toks(p))) >= 1
    def relevance(self, q, a):
        a2 = set(_toks(a)); return (len(set(_toks(q)) & a2) / len(a2)) if a2 else 0.0
class OfflineExtractor:
    def __init__(self, vocab): self.vocab = set(vocab)
    def extract(self, text): return [w for w in _toks(text) if w in self.vocab]
class ResilientReranker:
    def __init__(self, primary): self.primary, self._broken = primary, False
    def rerank(self, query, candidate_ids, passages):
        if not self._broken and self.primary is not None:
            try: return self.primary.rerank(query, candidate_ids, passages)
            except Exception as e:
                log.warning(f"NIM reranker failed ({e}); falling back to token-overlap."); self._broken = True
        q = set(_toks(query))
        return sorted(candidate_ids, key=lambda c: (-len(q & set(_toks(passages.get(c, "")))), c))

def make_generation(cfg, key):
    if cfg.GEN_BACKEND == "hf_local":
        try: return LocalHFClient(cfg.HF_MODEL_ID, cfg.MAX_NEW_TOKENS, cfg.TEMPERATURE, cfg.HF_LOAD_4BIT), f"hf_local:{cfg.HF_MODEL_ID}"
        except Exception as e: log.warning(f"hf_local failed ({e}); using NIM/offline.");
    if key and cfg.USE_NIM and cfg.GEN_BACKEND in ("nim", "hf_local"):
        return NimGenerationClient(cfg.NIM_BASE_URL, key, cfg.GEN_MODEL, cfg.NIM_RPM, cfg.TEMPERATURE, cfg.MAX_NEW_TOKENS), f"nim:{cfg.GEN_MODEL}"
    return OfflineGen(), "offline"

def build_services(cfg, key, topic_vocab):
    gen, gen_label = make_generation(cfg, key)
    if bool(key) and cfg.USE_NIM:
        from mgr.clients.nim import NimClient
        from mgr.clients.nim_adapters import NimEmbedder, NimReranker, NimGroundingJudge, NimEntityExtractor
        nim = NimClient(base_url=cfg.NIM_BASE_URL, api_key=key, rpm=cfg.NIM_RPM)
        return (gen, gen_label, NimEmbedder(nim, model=cfg.EMB_MODEL),
                ResilientReranker(NimReranker(nim, model=cfg.RERANK_MODEL)),
                NimGroundingJudge(nim, model=cfg.JUDGE_MODEL),
                NimEntityExtractor(nim, model=cfg.JUDGE_MODEL), "NIM")
    return (gen, gen_label, OfflineEmbedder(), ResilientReranker(None),
            OfflineJudge(), OfflineExtractor(topic_vocab), "OFFLINE")
print("Service factory ready.")

## 5 · NIM connectivity check (skipped offline)

In [ ]:
if HAVE_NIM:
    try:
        _g = NimGenerationClient(CFG.NIM_BASE_URL, NIM_KEY, CFG.GEN_MODEL, rpm=CFG.NIM_RPM, max_tokens=1)
        print("[OK] generation OK:", repr(_g.complete_text(CFG.GEN_MODEL, [{"role": "user", "content": "Reply with A."}])[0]))
        from mgr.clients.nim import NimClient
        d = NimClient(base_url=CFG.NIM_BASE_URL, api_key=NIM_KEY, rpm=CFG.NIM_RPM).embeddings(CFG.EMB_MODEL, ["myocardial infarction"])
        print(f"[OK] embeddings OK: dim={len(d['data'][0]['embedding'])}")
    except Exception as e:
        print("[FAIL] NIM check:", type(e).__name__, e, "\n  -> verify key/model names/Internet, or set CFG.USE_NIM=False.")
else:
    print("Offline mode - skipping NIM check.")

## 6 · Shared helpers

In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Image
def save_json(o, p): p = Path(p); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(json.dumps(o, indent=2, default=str)); return p
def load_json(p): return json.loads(Path(p).read_text())
def cached(p, force=None): force = CFG.FORCE if force is None else force; return (not force) and Path(p).exists()
def show_img(p): display(Image(str(p)))
print("Helpers ready.")

## 7 · Stage 1 — Manifest & budget (full 244-run contract preserved)

In [ ]:
from manifest.manifest import load_manifest, validate, budget_summary
M = load_manifest(); bs = budget_summary(M)
print(f"Manifest: {len(M)} runs | validation problems: {len(validate(M))}")
print(f"Full-program budget: ${bs['est_cost_usd']:.2f} base / ${bs['est_cost_usd_1.5x']:.2f} @1.5x")
print(f"POC executes {len(CFG.CONDITIONS)}x{len(CFG.SEEDS)}x{CFG.N_ITEMS} = "
      f"{len(CFG.CONDITIONS)*len(CFG.SEEDS)*CFG.N_ITEMS} generation calls")

## 8 · Stage 2 — Data (real MIRAGE subset + MedRAG textbook corpus)

`DATA_SOURCE="real"` fetches a **MIRAGE** question subset and a real **MedRAG medical-textbook**
corpus slice (no gold passage labels → retrieval quality is judged by RAGAS context-precision
downstream). `"synthetic"` (or any failure) uses a self-contained set *with* gold relevance.

> **StatPearls note:** `MedRAG/statpearls` on HF only ships a README — the actual StatPearls chunks
> require MedRAG's NCBI build script (licensing). We use `MedRAG/textbooks` (same corpus family:
> Harrison's Internal Medicine, First Aid, etc.). To use StatPearls at scale, run MedRAG's
> `statpearls` builder and point `CORPUS_HF`/`CORPUS_BOOK` at the produced JSONL.

In [ ]:
import json, random, urllib.request
from collections import Counter, defaultdict
data_root = DIRS["data"]
MIRAGE_TO_BENCH = {"mmlu": "MMLU-Med", "medqa": "MedQA-US", "medmcqa": "MedMCQA",
                   "pubmedqa": "PubMedQA", "bioasq": "BioASQ-YN"}
TOPIC_VOCAB = ["hypertension", "sepsis", "myocardial infarction", "asthma", "diabetes mellitus",
               "pneumonia", "stroke", "anemia", "hypothyroidism", "pulmonary embolism"]

def make_synth(n, seed):
    rng = random.Random(seed); L = "ABCD"; items, corpus, rel = [], [], {}
    for i in range(n):
        key, topic = f"q{i:04d}", rng.choice(TOPIC_VOCAB)
        opts = {L[j]: f"{key} choice{j} {key}c{j}n{rng.randrange(1000,9999)}" for j in range(4)}
        ci = rng.randrange(4)
        items.append({"qid": key, "question": f"For case {key} regarding {topic}, which option is supported by the record?",
                      "options": opts, "answer": L[ci]})
        pid = f"p{i:04d}"; corpus.append({"id": pid, "text": f"{topic} {key} {key} {opts[L[ci]]}", "topic": topic}); rel[key] = [pid]
    return items, corpus, rel

def _subject(qid):
    """MIRAGE qids are `<subject>-<n>` (e.g. `anatomy-000`)."""
    return str(qid).rsplit("-", 1)[0] if "-" in str(qid) else "all"

def stratify(records, n, seed):
    """Round-robin across subjects so a slice of `n` is not one single subject.

    MIRAGE's benchmark.json is grouped by subject, so `records[:n]` is a head
    slice of whichever subject happens to be first -- the 2026-07-13 run drew
    16 questions that were all `anatomy-*` and then asked them against an
    internal-medicine textbook.
    """
    buckets = defaultdict(list)
    for r in records:
        buckets[_subject(r.get("qid"))].append(r)
    rng = random.Random(seed)
    for b in buckets.values():
        rng.shuffle(b)
    out, subjects = [], sorted(buckets)
    while len(out) < n and any(buckets[s] for s in subjects):
        for s in subjects:
            if buckets[s]:
                out.append(buckets[s].pop())
                if len(out) == n:
                    break
    return out

def load_mirage(url, key, n, *, stratified=True, seed=0):
    from mgr.data.convert_mirage import convert_records
    raw = json.loads(urllib.request.urlopen(url, timeout=90).read().decode("utf-8"))
    if key not in raw: raise KeyError(f"{key} not in MIRAGE ({list(raw)})")
    recs = convert_records(raw[key])
    return stratify(recs, n, seed) if stratified else recs[:n]

def load_corpus_hf(hf_id, books, n_total, *, mode="spread", seed=0):
    """Sample `n_total` chunks spread over several books.

    `mode="spread"` reservoir-samples each book, so the slice is not the first
    pages (tables of contents, prefaces) of a single volume.
    """
    from huggingface_hub import hf_hub_download
    books = list(books)
    per_book = max(1, n_total // len(books))
    corpus, rng = [], random.Random(seed)
    for bi, book in enumerate(books):
        try:
            p = hf_hub_download(hf_id, f"chunk/{book}.jsonl", repo_type="dataset")
        except Exception as e:
            log.warning(f"corpus book {book!r} unavailable ({e}); skipping.")
            continue
        keep = []
        with open(p, encoding="utf-8") as fh:
            for i, line in enumerate(fh):
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                txt = (r.get("content") or r.get("contents") or "").strip()
                if not txt:
                    continue
                rec = {"id": f"{book}:{r.get('id', i)}", "text": txt,
                       "topic": str(r.get("title", book)), "book": book}
                if mode == "head":
                    keep.append(rec)
                    if len(keep) >= per_book:
                        break
                else:  # reservoir sample -> uniform over the whole book
                    if len(keep) < per_book:
                        keep.append(rec)
                    else:
                        j = rng.randrange(i + 1)
                        if j < per_book:
                            keep[j] = rec
        corpus.extend(keep)
        print(f"  [{bi+1}/{len(books)}] {book}: {len(keep)} chunks")
    return corpus

bench_path, corpus_path, rel_path = None, DIRS["data"] / "corpus.jsonl", DIRS["artifacts"] / "relevance.json"
RELEVANCE = None
if CFG.DATA_SOURCE == "real":
    try:
        CFG.BENCHMARK = MIRAGE_TO_BENCH[CFG.MIRAGE_DATASET]
        bench_path = DIRS["data"] / f"{CFG.BENCHMARK}.jsonl"
        if not (cached(bench_path) and cached(corpus_path)):
            items = load_mirage(CFG.MIRAGE_URL, CFG.MIRAGE_DATASET, CFG.N_ITEMS,
                                stratified=CFG.MIRAGE_STRATIFY, seed=CFG.SEED)
            bench_path.write_text("\n".join(json.dumps(x) for x in items))
            corpus = load_corpus_hf(CFG.CORPUS_HF, CFG.CORPUS_BOOKS, CFG.N_CORPUS_CHUNKS,
                                    mode=CFG.CORPUS_SAMPLE, seed=CFG.SEED)
            corpus_path.write_text("\n".join(json.dumps(x) for x in corpus))
            print(f"[real] MIRAGE {CFG.MIRAGE_DATASET}: {len(items)} q | corpus: {len(corpus)} chunks")
        else:
            print("[real] reusing cached real data.")
    except Exception as e:
        log.warning(f"real-data load failed ({e}); using synthetic.")
        CFG.DATA_SOURCE = "synthetic"

if CFG.DATA_SOURCE != "real":
    CFG.BENCHMARK = "MMLU-Med"; bench_path = DIRS["data"] / f"{CFG.BENCHMARK}.jsonl"
    if not (cached(bench_path) and cached(corpus_path) and cached(rel_path)):
        items, corpus, rel = make_synth(CFG.N_ITEMS, CFG.SEED)
        bench_path.write_text("\n".join(json.dumps(x) for x in items))
        corpus_path.write_text("\n".join(json.dumps(x) for x in corpus)); save_json(rel, rel_path)
    RELEVANCE = load_json(rel_path)
    print(f"[synthetic] {CFG.BENCHMARK}: gold-labelled set")

items = [json.loads(l) for l in bench_path.read_text().splitlines() if l]
corpus = [json.loads(l) for l in corpus_path.read_text().splitlines() if l]
passages = {c["id"]: c["text"] for c in corpus}

# Composition check: a single-subject question set against an unrelated corpus is
# not a retrieval experiment. Print it rather than discovering it after the run.
subj = Counter(_subject(it["qid"]) for it in items)
bookc = Counter(c.get("book", "?") for c in corpus)
print(f"\nbenchmark={CFG.BENCHMARK}  items={len(items)}  corpus_docs={len(corpus)}  "
      f"gold_relevance={'yes' if RELEVANCE else 'no (RAGAS ctx-precision used)'}")
print(f"question subjects ({len(subj)}): {dict(subj)}")
print(f"corpus books      ({len(bookc)}): {dict(bookc)}")
if len(subj) == 1 and CFG.DATA_SOURCE == "real":
    log.warning("All questions come from ONE subject — retrieval arms will be judged on a corpus "
                "that may not cover them. Set CFG.MIRAGE_STRATIFY=True.")
display(pd.DataFrame(items)[["qid", "question", "answer"]].head(4))

## 9 · Stage 3 — Instantiate services (NIM + chosen generation backend)

In [ ]:
GEN, GEN_LABEL, EMBEDDER, RERANKER, JUDGE, EXTRACTOR, MODE = build_services(CFG, NIM_KEY, TOPIC_VOCAB)
print(f"Service mode : {MODE}")
print(f"  generation : {GEN_LABEL}")
print(f"  embeddings : {CFG.EMB_MODEL if MODE=='NIM' else 'offline'}")
print(f"  reranker   : {CFG.RERANK_MODEL if MODE=='NIM' else 'token-overlap'}")
print(f"  judge/extr : {CFG.JUDGE_MODEL if MODE=='NIM' else 'offline'}")

## 10 · Stage 4 — Dense index via NIM embeddings (cached)

Embed the corpus once with `NimEmbedder`; persist the matrix to `cache/` so re-runs skip NIM.

In [ ]:
from mgr.retrieval.dense import DenseIndex, DenseRetriever
from mgr.retrieval.bm25 import BM25Index, BM25Retriever

emb_npy, ids_txt = DIRS["cache"] / "corpus_emb.npy", DIRS["cache"] / "corpus_ids.txt"
doc_ids = [c["id"] for c in corpus]
if cached(emb_npy) and cached(ids_txt) and ids_txt.read_text().split() == doc_ids:
    matrix = np.load(emb_npy); print("Loaded cached embeddings:", matrix.shape)
else:
    texts = [c["text"] for c in corpus]
    chunks = [np.asarray(EMBEDDER(texts[i:i+32]), dtype=float) for i in tqdm(range(0, len(texts), 32), desc="embed")]
    matrix = np.vstack(chunks); np.save(emb_npy, matrix); ids_txt.write_text("\n".join(doc_ids))
    print("Embedded corpus:", matrix.shape)

DENSE = DenseRetriever(DenseIndex.from_embeddings(doc_ids, matrix), EMBEDDER, passages)
BM25 = BM25Retriever(BM25Index.from_records(corpus))
print("Dense + BM25 retrievers ready.")

## 11 · Stage 5 — Grounded graph + **real UMLS** (scispaCy) → gate G3 + coverage (F5)

`UMLS_SOURCE="scispacy"` links entity mentions to real **UMLS CUIs** via scispaCy's UMLS knowledge
base, then feeds the repo's `UMLSLinker` (exact → abbrev → fuzzy) and emits the **coverage curve
(F5)** — the control for the "~47% ungrounded" defect, now on real concepts.

**G3 is now a real gate.** Previously the notebook printed *"gate G3 satisfied"* unconditionally and
set `GATE_LEDGER["G3"] = True` regardless of the number it had just computed — it reported 0.7%
coverage and a **1-concept, 1-link graph** and passed anyway. Downstream, an empty concept set makes
`ca_rrf(use_concept=True)` identical to plain RRF, so C2's ablation was guaranteed to show zero
effect and C3 had no concept signal to gate on. G3 now fails below `CFG.UMLS_COVERAGE_FLOOR`, and
the arms still run so you can see the damage — but the C2/C3 cells will tell you their results are
void.

In [ ]:
from mgr.graph.store import InMemoryGraphStore
from mgr.graph.build import build_graph, query_concept_fn
from mgr.graph.umls import UMLSLinker, coverage
from mgr.retrieval.graph_retriever import GraphRetriever
from mgr.figures import plot_coverage_curve

graph_corpus = corpus[:CFG.GRAPH_MAX_CHUNKS]

class ScispacyGrounder:
    """Real UMLS: scispaCy NER + UMLS linker. Provides mentions and a real surface->CUI dict.

    Entity extraction is memoized on the chunk text. The UMLS-linked pipeline costs
    roughly a second per chunk, and the notebook asks for the same chunks three
    times -- once in build_dict, once inside build_graph, once for the coverage
    curve. Without the cache, GRAPH_MAX_CHUNKS=800 would mean ~40 minutes of
    redundant CPU; with it, one pass.
    """
    def __init__(self, model):
        import spacy
        self.nlp = spacy.load(model, disable=["parser", "lemmatizer"])
        try: self.nlp.add_pipe("abbreviation_detector")
        except Exception: pass
        self.nlp.add_pipe("scispacy_linker",
                          config={"resolve_abbreviations": True, "linker_name": "umls", "max_entities_per_mention": 1})
        self.kb = self.nlp.get_pipe("scispacy_linker").kb
        self._ents = {}
    def _doc(self, text):
        return self.nlp(text[:4000])
    def extract(self, text):
        key = text[:4000]
        if key not in self._ents:
            self._ents[key] = [e.text for e in self._doc(key).ents]
        return self._ents[key]
    def build_dict(self, texts, max_concepts):
        exact, abbr = {}, {}
        for t in tqdm(texts, desc="scispaCy UMLS linking"):
            doc = self._doc(t[:4000])
            self._ents[t[:4000]] = [e.text for e in doc.ents]
            for e in doc.ents:
                if e._.kb_ents:
                    cui = e._.kb_ents[0][0]; exact[e.text.lower()] = cui
                    try: exact[self.kb.cui_to_entity[cui].canonical_name.lower()] = cui
                    except Exception: pass
            for ab in getattr(doc._, "abbreviations", []):
                try: abbr[str(ab).lower()] = str(ab._.long_form)
                except Exception: pass
            if len(exact) >= max_concepts: break
        return exact, abbr

if CFG.UMLS_SOURCE == "scispacy":
    try:
        _gr = ScispacyGrounder(CFG.SCISPACY_MODEL)
        exact_map, abbrev_map = _gr.build_dict([c["text"] for c in graph_corpus], CFG.UMLS_MAX_CONCEPTS)
        LINKER = UMLSLinker(exact_map=exact_map, abbrev_map=(abbrev_map or None), fuzzy_threshold=0.9)
        EXTRACTOR = _gr
        print(f"[scispacy] real UMLS dict: {len(exact_map)} surface forms, {len(abbrev_map)} abbreviations")
    except Exception as e:
        log.error(f"scispaCy UMLS unavailable ({e}); falling back to the toy dict. "
                  "C2 and C3 will NOT be measurable in this run — see the G3 verdict below.")
        CFG.UMLS_SOURCE = "toy"
if CFG.UMLS_SOURCE == "toy":
    LINKER = UMLSLinker(exact_map={t: f"C{ix:04d}" for ix, t in enumerate(TOPIC_VOCAB)}, fuzzy_threshold=0.9)

STORE = InMemoryGraphStore()
report = build_graph(graph_corpus, STORE, EXTRACTOR, LINKER)
QCF = query_concept_fn(EXTRACTOR, LINKER)
CANDIDATE_CONCEPTS = {cid: STORE.concepts_of(cid) for cid in passages}
GRAPH = GraphRetriever(STORE, QCF, hops=1)

all_mentions = []
for c in graph_corpus: all_mentions += EXTRACTOR.extract(c["text"])
cov = coverage(all_mentions, LINKER)

# ---- G3 is a go/no-go, not a print statement ------------------------------- #
# `curve` is cumulative, so the last tier IS the total linked fraction.
linked = cov.curve.get("exact+abbrev+fuzzy", 0.0)
concept_docs = sum(1 for s in CANDIDATE_CONCEPTS.values() if s)
G3_OK = (linked >= CFG.UMLS_COVERAGE_FLOOR) and (report.n_concepts >= 10) and (concept_docs > 0)
GATE_LEDGER = {"H2": True, "G3": bool(G3_OK), "P3": False}

save_json({"graph_hash": report.graph_hash, "n_chunks": report.n_chunks, "n_concepts": report.n_concepts,
           "n_links": report.n_links, "coverage": cov.curve, "umls_source": CFG.UMLS_SOURCE,
           "linked_frac": linked, "coverage_floor": CFG.UMLS_COVERAGE_FLOOR,
           "docs_with_concepts": concept_docs, "G3": bool(G3_OK)},
          DIRS["artifacts"] / "graph_report.json")

print(f"\nGraph: {report.n_chunks} chunks, {report.n_concepts} concepts, {report.n_links} links "
      f"| UMLS={CFG.UMLS_SOURCE}")
print(f"coverage curve: {dict((k, round(v,3)) for k,v in cov.curve.items())}")
print(f"docs carrying >=1 concept: {concept_docs}/{len(CANDIDATE_CONCEPTS)}")
if G3_OK:
    print(f"\n[G3 PASS] {linked:.1%} of mentions linked (floor {CFG.UMLS_COVERAGE_FLOOR:.0%}).")
else:
    print("\n" + "!" * 78)
    print(f"[G3 FAIL] only {linked:.1%} of mentions linked (floor {CFG.UMLS_COVERAGE_FLOOR:.0%}); "
          f"{report.n_concepts} concepts over {report.n_chunks} chunks.")
    print("The concept layer is effectively empty, so:")
    print("  - CA-RRF (C2) degenerates to plain RRF   -> its ablation reads 0/N by construction")
    print("  - CARe (C3) has no concept signal to gate on")
    print()
    print("Because G3 is now a REAL gate, Stage 7 will resolve every graph-touching row")
    print("to Blocked and run only No-RAG / BM25 / Dense-MedCPT. That is the gate doing")
    print("its job: the 2026-07-13 run forced G3=True and reported six arms whose graph")
    print("layer held exactly 1 concept over 60 chunks.")
    print("Blocked and run only No-RAG / BM25 / Dense-MedCPT. That is the gate working: the")
    print("2026-07-13 run forced G3=True and reported six arms whose graph layer held 1 concept.")
    print("Fix: keep UMLS_SOURCE='scispacy' and make sure §0 installed it BEFORE numpy was")
    print("imported (restart the kernel once if §0 asked you to).")
    print("!" * 78)
show_img(plot_coverage_curve(cov.curve, DIRS["figures"] / "F5_coverage.png"))

## 12 · Stage 6 — CARe oracle + gate fit → gate P3 (contribution C3, part 1)

The oracle label is **measured**, not assumed: for each query we build the context twice from the
*same* candidate pool — once as CA-RRF ranked it, once after the cross-encoder reranks it — generate
an answer for each, and label `oracle_benefit(correct_with, correct_without)`. That is what
`care_gate.py` documents ("labelled from B3 + M1 results") and it costs 2 generation calls per item.

The 2026-07-13 run instead labelled `1.0 if (near_tie_frac > 0.5 or top1_gap < 0.1) else 0.0` —
a threshold on two of the six features the gate is fit on, so the logistic model was being trained
to reproduce a deterministic function of its own inputs. It could not fail, and it measured nothing.

In [ ]:
from mgr.retrieval.fusion import build_fusion
from mgr.rerank.care_gate import extract_features, oracle_benefit, CareGate
from mgr.generate import prompts as _prompts
from mgr.generate.extract import normalize as _normalize
from mgr.generate.prompts import answer_type_for
from mgr.metrics.generation import exact_match as _exact_match

CARRF = build_fusion("Hybrid-CARRF", components={"lexical": BM25, "dense": DENSE, "graph": GRAPH},
                     passages=passages, query_concepts_fn=QCF, candidate_concepts=CANDIDATE_CONCEPTS)

class _SvcReranker:
    def rerank(self, q, ids, ps): return RERANKER.rerank(q, ids, ps)
CE_RERANKER = _SvcReranker()

ANSWER_TYPE = answer_type_for(M.benchmarks[CFG.BENCHMARK].get("type", "MCQ"))

def _answer_is_correct(it, ids):
    """Generate from exactly these passages and score against gold."""
    ctx = "\n\n".join(passages[i] for i in ids)
    msgs = _prompts.build_messages(it["question"], ANSWER_TYPE, options=it.get("options"), context=ctx)
    try:
        raw, _ = GEN.complete_text(CFG.GEN_MODEL, msgs)
    except Exception as e:
        log.warning(f"oracle generation failed for {it['qid']}: {type(e).__name__}: {e}")
        return 0.0
    return float(_exact_match(_normalize(raw, ANSWER_TYPE), it.get("answer")))

oracle_items = items[:CFG.CARE_ORACLE_ITEMS]
feats, labels, CARE_Q_WITH, CARE_Q_WO = [], [], [], []
n_rerank_changed = 0
for it in tqdm(oracle_items, desc="CARe oracle (rerank on/off)"):
    rr = CARRF.retrieve(it["question"], depth_k=10)
    feats.append(extract_features(rr.fused_scores or [0.0], query_type=0.0))

    ids_wo = list(rr.retrieved_ids)[:CFG.CTX_TOP_K]
    try:
        ids_with = CE_RERANKER.rerank(it["question"], list(rr.retrieved_ids), passages)[:CFG.CTX_TOP_K]
    except Exception as e:
        log.warning(f"rerank failed for {it['qid']}: {type(e).__name__}: {e}")
        ids_with = ids_wo
    n_rerank_changed += int(ids_with != ids_wo)

    q_wo = _answer_is_correct(it, ids_wo)
    q_with = q_wo if ids_with == ids_wo else _answer_is_correct(it, ids_with)
    CARE_Q_WO.append(q_wo); CARE_Q_WITH.append(q_with)
    labels.append(oracle_benefit(q_with, q_wo))

# A degenerate label set cannot train a gate. Say so instead of mirroring the
# labels to force a fit, which is what produced a meaningless "perfect" gate.
P3_OK = any(labels) and not all(labels)
if P3_OK:
    GATE = CareGate.fit(feats, labels)
else:
    log.warning(f"oracle labels are single-class ({sum(labels)}/{len(labels)} positive): reranking "
                "never changed an answer on this subset, so the gate is fit on a mirrored set and "
                "carries no information. Raise CARE_ORACLE_ITEMS or check that the reranker is live.")
    GATE = CareGate.fit(feats + feats, labels + [1 - x for x in labels])
GATE_LEDGER["P3"] = bool(P3_OK)
save_json(GATE_LEDGER, DIRS["artifacts"] / "gate_ledger.json")
save_json({"n_items": len(oracle_items), "positive_rate": float(np.mean(labels)),
           "rerank_changed_top_k": n_rerank_changed,
           "acc_with_rerank": float(np.mean(CARE_Q_WITH)),
           "acc_without_rerank": float(np.mean(CARE_Q_WO)), "P3": bool(P3_OK)},
          DIRS["artifacts"] / "care_oracle.json")

print(f"CARe oracle on {len(oracle_items)} items (measured, 2 generations each)")
print(f"  reranker changed the top-{CFG.CTX_TOP_K} context on {n_rerank_changed}/{len(oracle_items)} queries")
print(f"  accuracy with rerank   : {np.mean(CARE_Q_WITH):.3f}")
print(f"  accuracy without rerank: {np.mean(CARE_Q_WO):.3f}")
print(f"  oracle-positive rate   : {np.mean(labels):.2f}")
print(f"-> gate P3 {'satisfied' if P3_OK else 'NOT satisfied (single-class labels)'}")
if n_rerank_changed == 0:
    print("\n[!] The reranker never changed the ranking. Check §5 for a rerank 404 — a failed "
          "NIM reranker silently degrades to token overlap, which is what happened last run.")

## 13 · Stage 7 — Run the arms (real `Resources` -> `build_arm` -> runner)

In [ ]:
from mgr.sweep import Resources, build_arm
from mgr.runner import Runner
from mgr.tracking import record as rec
import shutil

RES = Resources(gen_client=GEN, data_root=str(DIRS["data"]), passages=passages,
                retrievers={"lexical": BM25, "dense": DENSE, "graph": GRAPH},
                query_concepts_fn=QCF, candidate_concepts=CANDIDATE_CONCEPTS,
                care_gate=GATE, reranker=CE_RERANKER, k=60, n_items=CFG.N_ITEMS)

arms_root = DIRS["results"] / "arms"
if CFG.FORCE and arms_root.exists(): shutil.rmtree(arms_root)
runner = Runner(M, GATE_LEDGER, str(arms_root)); records = []
for cond, seed in tqdm([(c, s) for c in CFG.CONDITIONS for s in CFG.SEEDS], desc="arms"):
    row = next((r for r in M.runs if r.condition == cond and r.benchmark == CFG.BENCHMARK
                and r.seed == seed and r.backbone == CFG.BACKBONE), None)
    if row is None: log.warning(f"no row for {cond}/s{seed}"); continue
    ex = rec.read_record(row.run_id, str(arms_root))
    if ex and ex.status == "Done" and not CFG.FORCE: records.append(ex); continue
    try:
        r = runner.run_one(row.run_id, executor=build_arm(cond, RES))
        if r: records.append(r)
    except Exception as e: log.error(f"{cond}/s{seed}: {type(e).__name__}: {e}")
runner.rollup()
print(f"Completed {len(records)} runs.")

## 14 · Stage 8 — Metrics per condition

Generation accuracy always; retrieval quality via **Recall@3** (synthetic gold) *or* **RAGAS
context-precision** from the NIM judge (real, gold-free data).

In [ ]:
from mgr.metrics import retrieval as rmetrics
from mgr.metrics.grounding_ragas import context_precision

RETR_FOR = {"BM25": BM25, "Dense-MedCPT": DENSE, "Graph-only": GRAPH,
            "Hybrid-CARRF": CARRF, "Hybrid-CARRF-CARe": CARRF}

# The executor now caps the retrieval query itself, but this stage calls
# retrieve() directly, so it keeps its own guard.
MAX_QUERY_CHARS = 6000

def _safe_retrieve(retriever, qid, question, depth_k):
    try:
        return retriever.retrieve(question[:MAX_QUERY_CHARS], depth_k=depth_k).retrieved_ids
    except Exception as e:
        log.warning(f"retrieve() failed for qid={qid!r} (len={len(question)} chars): {type(e).__name__}: {e}; using empty result")
        return []


def retrieval_axis():
    axis = {"No-RAG": 0.0}
    if RELEVANCE:
        for c, r in RETR_FOR.items():
            pairs = [(_safe_retrieve(r, it["qid"], it["question"], 10), RELEVANCE.get(it["qid"], [])) for it in items]
            axis[c] = rmetrics.score(pairs).recall.get(3, 0.0)
        return axis, "recall@3"
    for c, r in RETR_FOR.items():
        vals = []
        for it in items[:CFG.RAGAS_SUBSET]:
            ctx = [passages[i] for i in _safe_retrieve(r, it["qid"], it["question"], CFG.CTX_TOP_K)]
            vals.append(context_precision(it["question"], ctx, JUDGE))
        axis[c] = float(np.mean(vals)) if vals else 0.0
    return axis, "ctx_precision(RAGAS)"

RETR_AXIS, AXIS_NAME = retrieval_axis()
df = pd.DataFrame([{"condition": r.condition, "seed": r.seed, "n_items": r.n_items,
    "accuracy": round(r.metrics.get("generation", {}).get("accuracy", float('nan')), 3),
    "coverage": round(r.metrics.get("generation", {}).get("coverage", float('nan')), 3),
    # An item that raised is scored WRONG, not skipped: a transport blip is
    # otherwise indistinguishable from a model mistake. Last run, two arms each
    # lost one item this way and reported it only as status="Failed".
    "item_errors": r.metrics.get("generation", {}).get("n_item_errors", 0),
    AXIS_NAME: round(RETR_AXIS.get(r.condition, float('nan')), 3),
    "tokens": r.tokens.get("total"), "status": r.status} for r in records]).sort_values("condition")
df.to_csv(DIRS["artifacts"] / "arm_metrics.csv", index=False); display(df.reset_index(drop=True))

failed = [r for r in records if r.status != "Done" or r.error]
if failed:
    print("\n" + "=" * 78)
    print("ARMS THAT DID NOT COMPLETE CLEANLY — their accuracy is depressed by these errors:")
    for r in failed:
        print(f"  {r.condition:22s} status={r.status:8s} {r.error}")
    print("=" * 78)
else:
    print("\nAll arms completed with no per-item errors.")

# Sanity floor: with N items, one item is 100/N accuracy points. Say what is
# actually resolvable before anyone reads a ranking into this table.
_step = 100.0 / max(1, CFG.N_ITEMS)
print(f"\nResolution: N={CFG.N_ITEMS} -> 1 item = {_step:.1f} accuracy points. "
      f"Differences under ~{2*_step:.1f} points are within one or two items.")

## 15 · Stage 9 — RAGAS grounding via NIM judge (subset)

Two fixes since 2026-07-13, both of which depressed the previous scores (faithfulness 0.15,
answer relevance 0.38) for reasons that had nothing to do with grounding:

- the generation prompt contained the retrieved context and the words *"Answer briefly"* but **not
  the question** — the model was asked to answer nothing;
- it inherited `MAX_NEW_TOKENS=8`, sized for an MCQ letter, so any answer was truncated mid-phrase
  before the judge decomposed it into claims.

In [ ]:
from mgr.metrics.grounding_ragas import GroundingItem, score as gscore
sub = []
for it in tqdm(items[:CFG.RAGAS_SUBSET], desc="RAGAS"):
    rr = CARRF.retrieve(it["question"][:6000], depth_k=CFG.CTX_TOP_K)
    ctx = [passages[i] for i in rr.retrieved_ids]
    # The question has to be IN the prompt, and the answer needs room to be prose.
    msgs = [{"role": "user", "content":
             f"Context:\n{rr.context or ''}\n\nQuestion: {it['question']}\n\n"
             "Answer the question using only the context above. Two or three sentences."}]
    try:
        ans, _ = GEN.complete_text(CFG.GEN_MODEL, msgs, max_tokens=CFG.RAGAS_MAX_NEW_TOKENS)
    except Exception as e:
        log.warning(f"RAGAS generation failed for {it['qid']}: {type(e).__name__}: {e}")
        ans = ""
    sub.append(GroundingItem(question=it["question"], answer=ans or (ctx[0] if ctx else ""),
                             contexts=ctx, reference=it["options"].get(it["answer"], "") if it.get("options") else it.get("answer", "")))
gs = gscore(sub, JUDGE)
ragas = {"faithfulness": round(gs.faithfulness, 3), "answer_relevance": round(gs.answer_relevance, 3),
         "context_precision": round(gs.context_precision, 3), "context_recall": round(gs.context_recall, 3),
         "n_items": len(sub)}
save_json(ragas, DIRS["artifacts"] / "ragas.json"); display(pd.DataFrame([ragas]))
print("Sample answer (check it is prose about the question, not a truncated letter):")
print(" ", (sub[0].answer or "")[:300] if sub else "-")

## 16 · Stage 10 — Retrieval-Generation Decomposition · C1 (Figure F3)

In [ ]:
from mgr.stats.rgd import decompose, divergent_systems
from mgr.figures import plot_rgd
acc = {r.condition: r.metrics.get("generation", {}).get("accuracy", 0.0) for r in records}
systems = {c: (RETR_AXIS.get(c, 0.0), acc.get(c, 0.0)) for c in CFG.CONDITIONS if c in acc}
baseline = "No-RAG" if "No-RAG" in systems else list(systems)[0]
pts = decompose(systems, baseline=baseline)
display(pd.DataFrame([{"system": p.system, f"retrieval({AXIS_NAME})": round(p.retrieval, 3),
    "generation(acc)": round(p.generation, 3), "ret_gain": round(p.retrieval_gain, 3),
    "gen_gain": round(p.generation_gain, 3), "diverges": p.diverges} for p in pts]))
print("Divergent (retrieval up, generation flat/down):", divergent_systems(pts))
show_img(plot_rgd(pts, DIRS["figures"] / "F3_rgd.png", baseline=baseline))

## 17 · Stage 11 — CA-RRF ablation · C2 (isolable: concept-list on vs off)

In [ ]:
from mgr.retrieval.ca_rrf import ca_rrf
comp = {"lexical": BM25, "dense": DENSE, "graph": GRAPH}
changed, n_q_with_concepts = 0, 0
for it in items:
    q = it["question"][:6000]
    cl = {n: r.retrieve(q, depth_k=20).retrieved_ids for n, r in comp.items()}
    qc = set(QCF(q))
    n_q_with_concepts += int(bool(qc))
    on = [d for d, _ in ca_rrf(cl, qc, CANDIDATE_CONCEPTS, k=60, use_concept=True)]
    off = [d for d, _ in ca_rrf(cl, qc, CANDIDATE_CONCEPTS, k=60, use_concept=False)]
    if on[:5] != off[:5]: changed += 1
print(f"CA-RRF changed the top-5 vs plain RRF on {changed}/{len(items)} queries "
      "(use_concept=False is exactly plain RRF — the isolable ablation).")
print(f"queries with >=1 grounded concept: {n_q_with_concepts}/{len(items)}")

# A 0/N result means one of two very different things. Distinguish them.
if changed == 0:
    if n_q_with_concepts == 0 or not GATE_LEDGER.get("G3"):
        print("\n[C2 VOID] No query produced a grounded concept, so the concept list was empty and "
              "\nca_rrf(use_concept=True) is *identical by construction* to plain RRF. This is not a "
              "\nnull result about C2 — it is a null graph. See the G3 verdict in §11.")
    else:
        print("\n[C2 NULL] Concepts were present and CA-RRF still did not move the top-5. "
              "\nThat is a genuine (if small-N) negative result for the concept signal.")

## 18 · Stage 12 — CARe cost-quality frontier · C3 (Figure F4)

Built from the **measured** rerank-on / rerank-off outcomes collected in §12. The 2026-07-13 version
synthesised them (`wo = 1.0 if not amb else float(rng.random() < 0.45)`), which made `care` reach
`mean_quality = 1.000` by construction — F4 was a drawing of its own assumptions, not a measurement.

In [ ]:
from mgr.rerank.care_gate import cost_quality_frontier
from mgr.figures import plot_pareto

# q_with / q_wo come from §12's real generations — not from a random draw.
decisions = [GATE.decide(f, value=1.0, cost=0.15) for f in feats]
frontier = cost_quality_frontier(decisions, CARE_Q_WITH, CARE_Q_WO, rerank_cost=1.0)
display(pd.DataFrame([{"policy": k, "rerank_rate": round(v.rerank_rate, 3),
    "mean_quality": round(v.mean_quality, 3), "total_cost": v.total_cost} for k, v in frontier.items()]))
if not GATE_LEDGER.get("P3"):
    print("[C3 CAVEAT] P3 did not pass in §12 (single-class oracle labels), so this frontier is "
          "descriptive only — the gate carries no learned signal.")
show_img(plot_pareto(frontier, DIRS["figures"] / "F4_pareto.png"))

## 19 · Stage 13 — Statistics (audit -> CI -> exact-p -> Holm -> effect size)

In [ ]:
import json
from mgr.eval.answer_format_audit import audit
from mgr.stats.bootstrap import bootstrap_diff_ci
from mgr.stats.permutation import paired_permutation_test
from mgr.stats.holm import holm_bonferroni
from mgr.stats.effect_size import cohens_d_paired, cliffs_delta, interpret_cliffs
from mgr.generate.prompts import answer_type_for

def items_of(cond):
    r = next((r for r in records if r.condition == cond), None)
    return {it["qid"]: it for it in (json.loads(l) for l in open(r.items_path) if l.strip())} if (r and r.items_path) else {}

head = "Hybrid-CARRF-CARe" if any(r.condition == "Hybrid-CARRF-CARe" for r in records) else "BM25"
A, B = items_of(head), items_of("No-RAG"); qids = sorted(set(A) & set(B))
atype = answer_type_for(M.benchmarks[CFG.BENCHMARK].get("type", "MCQ"))
if qids:
    a = np.array([A[q]["correct"] for q in qids], float); b = np.array([B[q]["correct"] for q in qids], float)
    rep = audit({head: [A[q]["answer_norm"] for q in qids], "No-RAG": [B[q]["answer_norm"] for q in qids]}, answer_type=atype)
    print(f"{head} vs No-RAG on {len(qids)} items | audit passed: {rep.gate_emit_em_f1()} {rep.reasons}")
    if rep.gate_emit_em_f1():
        ci = bootstrap_diff_ci(a, b, n_boot=CFG.N_BOOT, seed=CFG.SEED)
        pr = paired_permutation_test(a, b, n_perm=CFG.N_PERM, seed=CFG.SEED)
        print(f"Delta acc={a.mean()-b.mean():+.3f}  95%CI[{ci.lo:+.3f},{ci.hi:+.3f}]")
        print(f"paired perm p={pr.p_value:.2e} exact={pr.exact} at_floor={pr.at_floor}")
        for h in holm_bonferroni({f"{head}_vs_NoRAG": pr.p_value}): print(f"Holm {h.label}: p_adj={h.p_adj:.2e} reject={h.reject}")
        print(f"Cohen's d={cohens_d_paired(a,b):.3f}  Cliff's delta={cliffs_delta(a,b):.3f} ({interpret_cliffs(cliffs_delta(a,b))})")
else:
    print("No overlapping qids to compare.")

## 20 · Figures gallery & run summary

In [ ]:
from pathlib import Path
for f in sorted(Path(DIRS["figures"]).glob("*.png")): show_img(f)
rows = [{"artifact": str(f.relative_to(RUN)), "bytes": f.stat().st_size}
        for d in DIRS.values() for f in Path(d).rglob("*") if f.is_file()]
display(pd.DataFrame(rows).sort_values("artifact").reset_index(drop=True))

# Provenance: the manifest row says BACKBONE, but generation actually went to
# GEN_LABEL. Last run every record was stamped `backbone=Llama-70B` while the
# answers came from llama-3.1-8b-instruct. Record both.
manifest_backbone = CFG.BACKBONE
actual_gen = GEN_LABEL
summary = {"mode": MODE, "generation_actual": actual_gen, "manifest_backbone": manifest_backbone,
           "backbone_matches_manifest": manifest_backbone.lower().replace("-", "") in actual_gen.lower().replace("-", ""),
           "data_source": CFG.DATA_SOURCE, "benchmark": CFG.BENCHMARK,
           "umls_source": CFG.UMLS_SOURCE, "conditions": list(CFG.CONDITIONS), "n_items": CFG.N_ITEMS,
           "gates": GATE_LEDGER, "n_runs": len(records),
           "n_arms_failed": sum(1 for r in records if r.status != "Done"),
           "run_dir": str(RUN)}
save_json(summary, DIRS["artifacts"] / "poc_summary.json")
print("Summary:", json.dumps(summary, indent=2, default=str))

print("\n" + "=" * 78)
print("VALIDITY LEDGER — read before quoting any number from this run")
print("=" * 78)
_verdicts = [
    ("G3 grounded graph", GATE_LEDGER.get("G3"), "C2 ablation and the graph arm are void without it"),
    ("P3 oracle labels",  GATE_LEDGER.get("P3"), "C3's gate carries no learned signal without it"),
    ("all arms Done",     summary["n_arms_failed"] == 0, "a failed arm's accuracy includes items scored wrong on transport errors"),
    ("backbone matches",  summary["backbone_matches_manifest"],
     f"records are stamped {manifest_backbone!r} but generation was {actual_gen!r}"),
]
for name, ok, why in _verdicts:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:22s} {'' if ok else '-> ' + why}")
_step = 100.0 / max(1, CFG.N_ITEMS)
print(f"\n  Resolution: 1 item = {_step:.1f} accuracy points at N={CFG.N_ITEMS}, {len(CFG.SEEDS)} seed(s).")
print("  A POC at this scale establishes that the pipeline runs. It does not establish an effect.")

## 21 · Scaling back up — compromises, sacrifices & restore paths

| # | POC compromise | Why | Sacrificed | Full-scale | Restore |
|---|---|---|---|---|---|
| 1 | **Generation: NIM-8B or local 3B** | No A100/vLLM free; 40 rpm | 70B answer quality | `llama-3.3-70b` on vLLM/A100 | `GEN_BACKEND` → point a `VLLMClient` at the A100; keep NIM for embed/rerank/judge |
| 2 | **~16 q / 1 seed** | runtime + rpm | statistical power, seed variance | full MIRAGE × 3–5 seeds | raise `N_ITEMS`,`SEEDS`; `run_sweep` over Ready rows |
| 3 | **~600-chunk 1-book corpus** | can't stage tens of GB | real retrieval breadth | full MedRAG corpora | raise `N_CORPUS_CHUNKS`; loop all `chunk/*.jsonl`; add PubMed/Wikipedia |
| 4 | **StatPearls → textbooks** | StatPearls not redistributed on HF | that specific corpus | StatPearls via MedRAG's NCBI builder | run MedRAG `statpearls` builder; point `CORPUS_HF/BOOK` at the JSONL |
| 5 | **scispaCy UMLS on ~60 chunks** | NER cost; ~1 GB KB | full-corpus concept coverage | scispaCy/QuickUMLS over all chunks + real `MRCONSO` | raise `GRAPH_MAX_CHUNKS`/`UMLS_MAX_CONCEPTS`; same linker |
| 6 | **In-memory graph** | no neo4j on Kaggle | persistence/scale | `Neo4jStore` (same protocol) | swap store; `infra/neo4j/*` |
| 7 | **CARe oracle from score dispersion** | doubling generations is costly | ground-truth oracle | labelled from B3+M1 outcomes | run CARRF vs staticRerank fully; `oracle_benefit(correct_with, without)` |
| 8 | **RAGAS on ~5 items** | judge-call cost | tight grounding CIs | full subset + human agreement | raise `RAGAS_SUBSET`; same `NimGroundingJudge` |
| 9 | **RunPod lifecycle omitted** | no cloud pods from Kaggle | metered auto-teardown sweep | on-demand A100 + guard | `docs/RUNBOOK.md`, `infra/runpod/*` |

**Mental model:** the POC changes *resources, data volume, and model size* — never the code paths.
Every service is injected through the same interface, so scaling up is a config/endpoint change.